<h1 style="text-align:center;">Song Success Prediction Model</h1>
<h2 style="text-align:center;">Preprocessing</h2>

<h4 style="text-align:center;">by Cameron Hicks</h3>

# Introduction

The Song Success Prediction Model project aims to create a Machine Learning Model that can predict if a song will be classified as a hit in the early or pre-release stages of production. The objective of this notebook is to prepare and process the combined Spotify Analytics and supplemental Country data so it is ready for Machiene Learning Modeling. Data Wrangling and Exploratory Data Analysis has been performed in previous notebooks in this project to properly combine and clean the data. Each step taken below is necessary to continue to the Model portion of this project.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load Data

In [2]:
model_df = pd.read_csv('../Data/V2_combined_df.csv')

In [3]:
model_df.head()

,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,...,lang_spanish,bloc_AU,bloc_EU,bloc_NAFTA,bloc_PA,bloc_USAN,hit,artist_song_count,days_since_release,streams_per_day
0,TRK-BEBD53DA84E1,Agent every (0),Noah Rhodes,Beautiful instead,2016-04-01,Pop,234194,55,0.15,0.74,...,False,False,False,False,False,True,0,1,3561,3.650660
1,TRK-6A32496762D7,Night respond,Jennifer Cole,Table,2022-04-15,Metal,375706,45,0.44,0.46,...,False,False,True,False,False,False,0,3,1356,0.737463
2,TRK-47AA7523463E,Future choice whatever,Brandon Davis,Page southern,2016-02-23,Rock,289191,55,0.62,0.80,...,False,False,True,False,False,False,0,2,3599,0.277855
3,TRK-25ADA22E3B06,Bad fall pick those,Corey Jones,Spring,2015-10-12,Pop,209484,51,0.78,0.98,...,False,False,True,False,False,False,0,4,3733,0.267881
4,TRK-9245F2AD996A,Husband,Mark Diaz,Great prove,2022-07-08,Indie,127435,39,0.74,0.18,...,False,False,False,True,False,False,0,2,1272,1.572327


In [4]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84954 entries, 0 to 84953
Data columns (total 49 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   track_id            84954 non-null  object 
 1   track_name          84954 non-null  object 
 2   artist_name         84954 non-null  object 
 3   album_name          84954 non-null  object 
 4   release_date        84954 non-null  object 
 5   genre               84954 non-null  object 
 6   duration_ms         84954 non-null  int64  
 7   popularity          84954 non-null  int64  
 8   danceability        84954 non-null  float64
 9   energy              84954 non-null  float64
 10  key                 84954 non-null  int64  
 11  loudness            84954 non-null  float64
 12  mode                84954 non-null  object 
 13  instrumentalness    84954 non-null  float64
 14  tempo               84954 non-null  float64
 15  stream_count        84954 non-null  int64  
 16  coun

In [5]:
# Identify high-cardinality categorical columns
obj_unique_counts = model_df.select_dtypes(include=['object']).nunique()

obj_unique_counts

track_id             84954
track_name           68915
artist_name          62364
album_name           43170
release_date          4018
genre                   12
mode                     2
country_name            10
label                    8
duration_mm:ss:ms    75045
alpha2Code              10
capital                 10
subregion                9
region                   5
latlng                  10
demonym                 10
currency_code            8
dtype: int64

# Drop Unnecessary Columns

Several of the features of the data were created or left in place for Exploratory Data Analysis purposes. Additionally, several of the categorical columns have a high number of unique values which can pose a problem when one-hot encoding. These columns must be dropped from the dataset prior to modeling to prevent risk of leakage, noise, and overfitting the model.

- track_id - This is a unique ID used to identify the track in the dataframe. It is not the name of the song that will be released and will not be public knowledge.
- track_name - This is the actual name of each song and has a very high number of unique values.
- artist_name - In EDA we discovered that the dataset is not heavily influenced by artist name for songs, the majority of artists only have one song and therefore one record. Due to potential high-cardinality issues, this column will be dropped.
- album_name - In the EDA phase we found that most albums only contain one song so there are a very high number of unique values.
- duration_mm:ss:ms - This column contains the same information as duration_ms only in a different format that is easier to read. It was created only to improve the EDA process.
- alpha2Code - This is duplicate data to the Country Name and poses risk to Data Leakage.
- latlng - This column was kept for EDA, it contains a list version of the latitue and longitude of each country. It is no longer needed since we have the country name.
- capital - since our dataset does not contain insights of number of streams or popularity per city, Capital is essentially a duplicate feature of Country
- demonym - This feature is redundant with Country
- streams_per_day - This column was designed to help visualize the actual streaming ratio against popularity. It poses a major risk to data leakage and must be removed.

In [6]:
cols_to_drop = ['track_id', 'track_name', 'artist_name', 'album_name', 'duration_mm:ss:ms', 'alpha2Code', 'latlng', 'capital', 'demonym', 'streams_per_day']

model_df = model_df.drop(columns=cols_to_drop, axis=1)

We also must drop stream_count. In the Data Wrangling and EDA steps we explored this feature and flagged it as potentially the target feature against popularity. It has been determined that popularity is a stronger target feature due to indicating popularity regardless of when the song was released while stream_count can be skewed by the length of time availability to stream. For this reason stream_count will be dropped to prevent leakage and overfitting.

In [7]:
model_df = model_df.drop('stream_count', axis=1)

Finally we must drop the popularity column. The objective of the model is to determine if a song will be classified as a hit. The hit column was built based on if the popularity score was 70 or higher. This column must be droped to avoid leakage and overfitting.

In [8]:
model_df = model_df.drop('popularity', axis=1)

In [9]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84954 entries, 0 to 84953
Data columns (total 37 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   release_date        84954 non-null  object 
 1   genre               84954 non-null  object 
 2   duration_ms         84954 non-null  int64  
 3   danceability        84954 non-null  float64
 4   energy              84954 non-null  float64
 5   key                 84954 non-null  int64  
 6   loudness            84954 non-null  float64
 7   mode                84954 non-null  object 
 8   instrumentalness    84954 non-null  float64
 9   tempo               84954 non-null  float64
 10  country_name        84954 non-null  object 
 11  explicit            84954 non-null  bool   
 12  label               84954 non-null  object 
 13  subregion           84954 non-null  object 
 14  region              84954 non-null  object 
 15  population          84954 non-null  int64  
 16  area

# Feature Engineering

In [35]:
# Replace 'release_date' with the number of days since release. This will be more useful in model training

model_df['release_date'] = pd.to_datetime(model_df['release_date'])

model_df['days_since_release'] = (pd.Timestamp.today() - model_df['release_date']).dt.days

In [36]:
model_df['days_since_release'].head()

0    3611
1    1406
2    3649
3    3783
4    1322
Name: days_since_release, dtype: int64

In [37]:
model_df = model_df.drop('release_date', axis=1)

In [38]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84954 entries, 0 to 84953
Data columns (total 37 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   genre               84954 non-null  object 
 1   duration_ms         84954 non-null  int64  
 2   danceability        84954 non-null  float64
 3   energy              84954 non-null  float64
 4   key                 84954 non-null  int64  
 5   loudness            84954 non-null  float64
 6   mode                84954 non-null  object 
 7   instrumentalness    84954 non-null  float64
 8   tempo               84954 non-null  float64
 9   country_name        84954 non-null  object 
 10  explicit            84954 non-null  bool   
 11  label               84954 non-null  object 
 12  alpha2Code          84954 non-null  object 
 13  subregion           84954 non-null  object 
 14  region              84954 non-null  object 
 15  population          84954 non-null  int64  
 16  area

In [39]:
# separate features by their type for feature engineering

cat_cols = model_df.select_dtypes(include='object').columns
num_cols = model_df.select_dtypes(include=['int64','float64']).columns
bool_cols = model_df.select_dtypes(include='bool').columns

## Crete Dummy Variables (One-Hot Encoding)

In [40]:
model_df = pd.get_dummies(model_df, columns=cat_cols, drop_first=True)

In [41]:
# Several boolean columns were created in the Data Wrangling step. These are already one-hot encoded, but need to be converted to numeric

model_df[bool_cols] = model_df[bool_cols].astype(int)

In [42]:
model_df.shape

(84954, 85)

In [43]:
model_df.to_csv('../Data/V2_model_df_processed.csv', index=False)

# Train Test Split

In [44]:
# Separate data into X = all columns except target feature & y = target feature

X = model_df.drop(columns='hit')
y = model_df['hit']

In [45]:
X.shape

(84954, 84)

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [47]:
X_train.head()

,duration_ms,danceability,energy,key,loudness,instrumentalness,tempo,explicit,population,area,...,region_asia,region_europe,region_oceania,currency_code_BRL,currency_code_CAD,currency_code_EUR,currency_code_GBP,currency_code_JPY,currency_code_MXN,currency_code_USD
53923,107007,0.65,0.23,2,-36.57,0.104,69.45,0,125836021,377930,...,True,False,False,False,False,False,False,True,False,False
64425,346730,0.52,0.60,5,-48.25,0.605,171.20,0,25687041,7692024,...,False,False,True,False,False,False,False,False,False,False
24324,122942,0.60,0.24,6,-31.66,0.428,183.30,0,212559409,8515767,...,False,False,False,True,False,False,False,False,False,False
25685,287498,0.32,0.11,5,-27.83,0.768,96.03,0,212559409,8515767,...,False,False,False,True,False,False,False,False,False,False
2964,316116,0.38,0.54,10,-24.86,0.595,169.84,1,3000,60,...,False,False,False,False,False,False,False,False,False,True


# Scale Features

In [48]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Preprocessing Conclusion

The data has been fully preprocessed and is ready for modeling. Steps taken in the preprocessing phase were:

- Dropped Unnecessary columns - columns were removed if they were identifiers, redundant representations of existing variables, high-cardinality text fields, or created for EDA only.
- Replaced Release Date column with Days Since Release to allow for better model interpretation
- Separated features by type into three classifications - Categorical, Numberical, and Boolean. Each of these needs to be processed differently to prepare for training
- Created Dummy Variables / One-Hot Encoding on categorical features using pd.get_dummies(). One category from each feature was dropped to avoid multicollinearity.
- Split Data for Training and Testing - All features were separated from the target feature (popularity) into the variable X while the target feature was placed in the variable y. Then the data was split into training and test sets using sklearn's train_test_split with the standard 80/20 ratio.
- Scaled Features - Feature scaling was applied after the train-test split to prevent data leakage. The scaler was fit only on the training data and then applied to both the training and test datasets to ensure that the model evaluation reflects real-world performance.

In [49]:
pd.DataFrame(X_train_scaled).to_csv('../Data/V2_X_train_scaled.csv', index=False)
pd.DataFrame(X_test_scaled).to_csv('../Data/V2_X_test_scaled.csv', index=False)
y_train.to_csv('../Data/V2_y_train.csv', index=False)
y_test.to_csv('../Data/V2_y_test.csv', index=False)